In [1]:
# Cell 1: Imports & State Definition
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END

# FraudShield Agent State
# This is shared memory — every agent read/writes it
class FraudShieldState(TypedDict):
    transaction_id: str
    amount: float
    risk_score: float
    document_verified: bool
    compliance_passed: bool
    case_notes: str
    decision: Literal["approve", "reject", "escalate_to_human"]
    human_review_needed: bool

print("✅ State defined successfully!")


✅ State defined successfully!


In [2]:
# Cell 2: Document Verification Agent
def document_verification_agent(state: FraudShieldState) -> FraudShieldState:
    print(f"🔍 Verifying document for Transaction: {state['transaction_id']}")

    # Rule: amount > 100000 needs extra verification
    if state['amount'] > 100000:
        state['document_verified'] = False
        state['case_notes'] = "High value transaction - document verification failed."
    else:
        state['document_verified'] = True
        state['case_notes'] = "Document verification passed."

    print(f"📄 Document Verified: {state['document_verified']}")
    return state

print("✅ Document Verification Agent defined!")


✅ Document Verification Agent defined!


In [3]:
# Cell 3: Risk Scoring Agent
def risk_scoring_agent(state: FraudShieldState) -> FraudShieldState:
    print(f"⚡ Calculating Risk Score for: {state['transaction_id']}")

# Simulate XGBoost model output
# Real : model.predict_proba([[features]])[0][1]
    if state['amount'] > 50000:
        state['risk_score'] = 0.85 # High risk
    elif state['amount'] > 10000:
        state['risk_score'] = 0.45 # Medium risk
    else:
        state['risk_score'] = 0.12 # Low risk

# Human review needed if risk > 0.7
    if state['risk_score'] > 0.7:
        state['human_review_needed'] = True
        state['case_notes'] += f" | High risk score: {state['risk_score']}"
    else:
        state['human_review_needed'] = False

    print(f"🎯 Risk Score: {state['risk_score']} | Human Review: {state['human_review_needed']}")
    return state

print("✅ Risk Scoring Agent defined!")


✅ Risk Scoring Agent defined!


In [4]:
# Cell 4: Compliance Agent
def compliance_agent(state: FraudShieldState) -> FraudShieldState:
    print(f"📋 Running Compliance Check for: {state['transaction_id']}")

# Rule: If document not verified OR risk too high → fail compliance
    if not state['document_verified'] or state['risk_score'] > 0.8:
        state['compliance_passed'] = False
        state['decision'] = "reject"
        state['case_notes'] += " | Compliance FAILED - rejected."
    elif state['human_review_needed']:
        state['compliance_passed'] = True
        state['decision'] = "escalate_to_human"
        state['case_notes'] += " | Escalated to human analyst."
    else:
        state['compliance_passed'] = True
        state['decision'] = "approve"
        state['case_notes'] += " | Compliance passed - approved."

    print(f"✅ Compliance: {state['compliance_passed']} | Decision: {state['decision']}")
    return state

print("✅ Compliance Agent defined!")

✅ Compliance Agent defined!


In [6]:
# Cell 5: Case Notes Agent
def case_notes_agent(state: FraudShieldState) -> FraudShieldState:
    print(f"📝 Writing Case Notes for: {state['transaction_id']}")
    summary = "\n================================\n"
    summary += "FRAUDSHIELD CASE REPORT\n"
    summary += "================================\n"
    summary += f"Transaction ID : {state['transaction_id']}\n"
    summary += f"Amount : Rs.{state['amount']}\n"
    summary += f"Risk Score : {state['risk_score']}\n"
    summary += f"Doc Verified : {state['document_verified']}\n"
    summary += f"Compliance : {state['compliance_passed']}\n"
    summary += f"Final Decision : {state['decision'].upper()}\n"
    summary += f"Human Review : {state['human_review_needed']}\n"
    summary += "--------------------------------\n"
    summary += f"Notes: {state['case_notes']}\n"
    summary += "================================"
    state['case_notes'] = summary
    print(summary)
    return state

print("✅ Case Notes Agent defined!")



✅ Case Notes Agent defined!


In [7]:
# Cell 6: Build the LangGraph
workflow = StateGraph(FraudShieldState)

# Add all 4 agents as nodes
workflow.add_node("document_verification", document_verification_agent)
workflow.add_node("risk_scoring", risk_scoring_agent)
workflow.add_node("compliance", compliance_agent)
workflow.add_node("case_notes", case_notes_agent)

# Connect nodes in sequence
workflow.set_entry_point("document_verification")
workflow.add_edge("document_verification", "risk_scoring")
workflow.add_edge("risk_scoring", "compliance")
workflow.add_edge("compliance", "case_notes")
workflow.add_edge("case_notes", END)

# Compile the graph
app = workflow.compile()
print("✅ FraudShield Graph compiled successfully!")


✅ FraudShield Graph compiled successfully!


In [8]:
# Cell 7: Run the FraudShield Pipeline!
test_transaction = {
"transaction_id": "TXN_001",
"amount": 75000.0,
"risk_score": 0.0,
"document_verified": False,
"compliance_passed": False,
"case_notes": "",
"decision": "approve",
"human_review_needed": False
}

print("🚀 Running FraudShield Multi-Agent Pipeline...")
print("=" * 40)

result = app.invoke(test_transaction)

print("\n🎯 FINAL RESULT:")
print(f"Decision: {result['decision'].upper()}")
print(f"Human Review Needed: {result['human_review_needed']}")


🚀 Running FraudShield Multi-Agent Pipeline...
🔍 Verifying document for Transaction: TXN_001
📄 Document Verified: True
⚡ Calculating Risk Score for: TXN_001
🎯 Risk Score: 0.85 | Human Review: True
📋 Running Compliance Check for: TXN_001
✅ Compliance: False | Decision: reject
📝 Writing Case Notes for: TXN_001

FRAUDSHIELD CASE REPORT
Transaction ID : TXN_001
Amount : Rs.75000.0
Risk Score : 0.85
Doc Verified : True
Compliance : False
Final Decision : REJECT
Human Review : True
--------------------------------
Notes: Document verification passed. | High risk score: 0.85 | Compliance FAILED - rejected.

🎯 FINAL RESULT:
Decision: REJECT
Human Review Needed: True


In [10]:
# Cell 8: Load Real Data + Model
import pandas as pd
import numpy as np
import joblib

# Load dataset
df = pd.read_csv(r"C:\Users\Abcom\Desktop\Fraudshield\creditcard.csv")

# Load saved XGBoost model
model = joblib.load(r"C:\Users\Abcom\Desktop\Fraudshield\fraudshield_model.pkl")

print(f"✅ Dataset loaded: {df.shape}")
print(f"✅ Model loaded successfully!")
print(f"Sample transaction columns: {list(df.columns[:5])}")



✅ Dataset loaded: (284807, 31)
✅ Model loaded successfully!
Sample transaction columns: ['Time', 'V1', 'V2', 'V3', 'V4']


In [14]:
# Cell 9: Check exact model features
print("Model expects these features:")
print(model.feature_names_in_)
print(f"\nTotal features: {len(model.feature_names_in_)}")


Model expects these features:
['V1' 'V2' 'V3' 'V4' 'V5' 'V6' 'V7' 'V8' 'V9' 'V10' 'V11' 'V12' 'V13'
 'V14' 'V15' 'V16' 'V17' 'V18' 'V19' 'V20' 'V21' 'V22' 'V23' 'V24' 'V25'
 'V26' 'V27' 'V28' 'Amount_scaled' 'Time_scaled']

Total features: 30


In [ ]:
# Cell 10: Pick Real Transaction + Get Actual Risk Score
from sklearn.preprocessing import StandardScaler

#  Create scaled columns 
scaler_amount = StandardScaler()
scaler_time = StandardScaler()

df['Amount_scaled'] = scaler_amount.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler_time.fit_transform(df[['Time']])

#  Take Fraud transaction
fraud_tx = df[df['Class'] == 1].iloc[0]

# Exact model features
feature_cols = list(model.feature_names_in_)
features_df = pd.DataFrame([fraud_tx[feature_cols]], columns=feature_cols)

# Risk score
risk_score = model.predict_proba(features_df)[0][1]
amount = fraud_tx['Amount']

print(f"💳 Real Transaction Amount: Rs.{amount:.2f}")
print(f"🎯 XGBoost Risk Score: {round(risk_score, 2)}")
print(f"🚨 Actual Label: FRAUD")


💳 Real Transaction Amount: Rs.0.00
🎯 XGBoost Risk Score: 1.0
🚨 Actual Label: FRAUD


In [16]:
# Cell 11: Run Real Transaction through LangGraph Pipeline
real_transaction = {
"transaction_id": "TXN_REAL_001",
"amount": float(amount),
"risk_score": float(risk_score),
"document_verified": False,
"compliance_passed": False,
"case_notes": "",
"decision": "approve",
"human_review_needed": False
}

print("🚀 Running REAL transaction through FraudShield Pipeline...")
print("=" * 40)

result = app.invoke(real_transaction)

print("\n🎯 FINAL RESULT:")
print(f"Decision: {result['decision'].upper()}")
print(f"Human Review Needed: {result['human_review_needed']}")



🚀 Running REAL transaction through FraudShield Pipeline...
🔍 Verifying document for Transaction: TXN_REAL_001
📄 Document Verified: True
⚡ Calculating Risk Score for: TXN_REAL_001
🎯 Risk Score: 0.12 | Human Review: False
📋 Running Compliance Check for: TXN_REAL_001
✅ Compliance: True | Decision: approve
📝 Writing Case Notes for: TXN_REAL_001

FRAUDSHIELD CASE REPORT
Transaction ID : TXN_REAL_001
Amount : Rs.0.0
Risk Score : 0.12
Doc Verified : True
Compliance : True
Final Decision : APPROVE
Human Review : False
--------------------------------
Notes: Document verification passed. | Compliance passed - approved.

🎯 FINAL RESULT:
Decision: APPROVE
Human Review Needed: False
